# User Behaviour Prediction System Using Machine Learning
### B.Tech Final Year Project

**Dataset :** UCI Online Shoppers Purchasing Intention Dataset
**Algorithm :** Logistic Regression (L2 regularisation, L-BFGS solver)

---
**How this notebook is organised:**
| Notebook Section | Description |
|---|---|
| 1. Setup & Imports | — |
| 2. Load Dataset | Load UCI dataset (12,330 records) |
| 3. EDA | PageValues, monthly trends, visitor types |
| 4. Preprocessing | Null drop → encode → scale → SMOTE |
| 5. Model Training | LogisticRegression (L2, lbfgs, C=1.0) |
| 6. Evaluation | Accuracy, F1, AUC, confusion matrix, ROC |
| 7. Feature Importance | Top 20 coefficients |
| 8. Save Model | Export model + scaler as .pkl |

---
## Section 1 — Setup & Imports

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# Uncomment the line below if you haven't installed these yet:
# !pip install scikit-learn imbalanced-learn pandas numpy matplotlib seaborn joblib

# ── Core libraries ────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import joblib
import os

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
 classification_report, confusion_matrix, accuracy_score,
 roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

# ── Imbalanced-learn (SMOTE) ──────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE

# ── Settings ──────────────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
RANDOM_STATE = 42 # Fixed seed — ensures reproducible results every run
TEST_SIZE = 0.20 # 80% train / 20% test

print('All libraries imported successfully.')
print(f'Random state : {RANDOM_STATE}')
print(f'Test size : {TEST_SIZE*100:.0f}%')

---
## Section 2 — Load Dataset
> **

In [ ]:
# ── Download dataset directly from UCI ───────────────────────────────────────
# The dataset is hosted publicly — no Kaggle account needed.
URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00468/online_shoppers_intention.csv'

# If you have no internet, place the CSV in the same folder as this notebook
# and change URL to: 'online_shoppers_intention.csv'
try:
 df = pd.read_csv(URL)
 print('Dataset downloaded from UCI.')
except Exception:
 df = pd.read_csv('online_shoppers_intention.csv')
 print('Dataset loaded from local file.')

print(f'\nShape : {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# ── Basic information ─────────────────────────────────────────────────────────
print('=== COLUMN TYPES ===')
print(df.dtypes)
print()
print('=== MISSING VALUES ===')
print(df.isnull().sum())
print(f'\nTotal missing values : {df.isnull().sum().sum()}')

In [ ]:
# ── Class distribution ──────────────────────────────────

class_counts = df['Revenue'].value_counts()
print('=== TARGET CLASS DISTRIBUTION (Revenue) ===')
print(f"No Purchase (False) : {class_counts[False]:,} ({class_counts[False]/len(df)*100:.1f}%)")
print(f"Purchase (True) : {class_counts[True]:,} ({class_counts[True]/len(df)*100:.1f}%)")
print(f"Class imbalance ratio : {class_counts[False]/class_counts[True]:.1f} : 1")

---
## Section 3 — Exploratory Data Analysis (EDA)
> **

In [ ]:
# ── Plot 1: Class distribution ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = ['#5B8DB8', '#E07B54']
axes[0].bar(['No Purchase\n(False)', 'Purchase\n(True)'],
 [class_counts[False], class_counts[True]],
 color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Target Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Sessions')
for i, v in enumerate([class_counts[False], class_counts[True]]):
 axes[0].text(i, v + 100, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=10)
axes[0].set_ylim(0, 12500)

# Pie chart
axes[1].pie([class_counts[False], class_counts[True]],
 labels=['No Purchase (84.5%)', 'Purchase (15.5%)'],
 colors=colors, autopct='%1.1f%%', startangle=140,
 wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Purchase vs Non-Purchase Split', fontsize=13, fontweight='bold')

plt.suptitle('Fig 1: Class Imbalance in Online Shoppers Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('fig1_class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig1_class_distribution.png')

In [ ]:
# ── Plot 2: PageValues by Revenue ─────────────────────────────────────────────

# Page Values (mean=21.5) compared to non-purchase sessions (mean=0.89)"
pv_purchase = df[df['Revenue'] == True]['PageValues']
pv_nopurchase = df[df['Revenue'] == False]['PageValues']

print('=== PAGE VALUES STATISTICS ===')
print(f'Purchase sessions — mean: {pv_purchase.mean():.2f} | median: {pv_purchase.median():.2f}')
print(f'Non-purchase sessions — mean: {pv_nopurchase.mean():.2f} | median: {pv_nopurchase.median():.2f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pv_nopurchase[pv_nopurchase < 100], bins=40, alpha=0.7,
 color='#5B8DB8', label=f'No Purchase (mean={pv_nopurchase.mean():.2f})')
ax.hist(pv_purchase[pv_purchase < 100], bins=40, alpha=0.7,
 color='#E07B54', label=f'Purchase (mean={pv_purchase.mean():.2f})')
ax.set_xlabel('Page Value (clipped at 100 for visibility)')
ax.set_ylabel('Number of Sessions')
ax.set_title('Fig 2: PageValues Distribution by Purchase Outcome\n(Key finding — )',
 fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('fig2_pagevalues.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig2_pagevalues.png')

In [ ]:
# ── Plot 3: Monthly purchase rate ─────────────────────────────────────────────
#
month_order = ['Feb', 'Mar', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly = df.groupby('Month')['Revenue'].agg(['sum', 'count'])
monthly['purchase_rate'] = monthly['sum'] / monthly['count'] * 100
monthly = monthly.reindex([m for m in month_order if m in monthly.index])

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(monthly.index, monthly['purchase_rate'],
 color=['#E07B54' if m == 'Nov' else '#5B8DB8' for m in monthly.index],
 edgecolor='white', linewidth=1.2)
ax.set_xlabel('Month')
ax.set_ylabel('Purchase Rate (%)')
ax.set_title('Fig 3: Purchase Rate by Month\n(November peak — )',
 fontsize=12, fontweight='bold')
for bar, val in zip(bars, monthly['purchase_rate']):
 ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
 f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('fig3_monthly_purchase_rate.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig3_monthly_purchase_rate.png')

In [ ]:
# ── Plot 4: Visitor type analysis ─────────────────────────────────────────────
#
visitor_stats = df.groupby('VisitorType')['Revenue'].agg(['count', 'sum'])
visitor_stats.columns = ['Total Sessions', 'Purchases']
visitor_stats['Purchase Rate (%)'] = (visitor_stats['Purchases'] / visitor_stats['Total Sessions'] * 100).round(2)
visitor_stats['% of Total Sessions'] = (visitor_stats['Total Sessions'] / len(df) * 100).round(2)
visitor_stats['% of Total Purchases'] = (visitor_stats['Purchases'] / df['Revenue'].sum() * 100).round(2)

print('=== VISITOR TYPE ANALYSIS ===')
print(visitor_stats.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(visitor_stats))
w = 0.35
ax.bar(x - w/2, visitor_stats['% of Total Sessions'], w,
 label='% of Total Sessions', color='#5B8DB8', edgecolor='white')
ax.bar(x + w/2, visitor_stats['% of Total Purchases'], w,
 label='% of Total Purchases', color='#E07B54', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(visitor_stats.index)
ax.set_ylabel('Percentage (%)')
ax.set_title('Fig 4: Visitor Type — Sessions vs Purchase Share\n',
 fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('fig4_visitor_type.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig4_visitor_type.png')

In [ ]:
# ── Plot 5: Correlation heatmap (numeric features) ────────────────────────────
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
df_num = df[numeric_cols].copy()
df_num['Revenue'] = df['Revenue'].astype(int)

fig, ax = plt.subplots(figsize=(11, 8))
corr = df_num.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
 center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8},
 annot_kws={'size': 8})
ax.set_title('Fig 5: Correlation Matrix — Numeric Features\n(Preprocessing insight)',
 fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig5_correlation_heatmap.png')

---
## Section 4 — Preprocessing Pipeline
> **> 1. Drop 14 null rows
> 2. One-hot encode categorical columns
> 3. StandardScaler on all features
> 4. 80/20 stratified split
> 5. SMOTE on training set only (no leakage)

In [ ]:
# ── Step 1: Drop missing values ───────────────────────────────────────────────
#
print(f'Rows before dropping nulls : {len(df):,}')
df_clean = df.dropna().copy()
print(f'Rows after dropping nulls : {len(df_clean):,}')
print(f'Rows removed : {len(df) - len(df_clean)}')

# ── Step 2: Separate features and target ─────────────────────────────────────
X = df_clean.drop('Revenue', axis=1)
y = df_clean['Revenue'].astype(int) # True → 1 (purchase), False → 0

print(f'\nFeatures (X) shape : {X.shape}')
print(f'Target (y) shape : {y.shape}')
print(f'\nClass counts in y : 0={sum(y==0):,} 1={sum(y==1):,}')

In [ ]:
# ── Step 3: Categorical encoding ──────────────────────────────────────────────

# OperatingSystems) are transformed using one-hot encoding"
# Weekend is already boolean — cast to int

categorical_cols = ['Month', 'VisitorType', 'Browser', 'Region',
 'TrafficType', 'OperatingSystems']

X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=False)
X_encoded['Weekend'] = X_encoded['Weekend'].astype(int)

print(f'Features before encoding : {X.shape[1]}')
print(f'Features after encoding : {X_encoded.shape[1]}')
print(f'\nAll encoded feature names:')
for i, col in enumerate(X_encoded.columns, 1):
 print(f' {i:2d}. {col}')

In [ ]:
# ── Step 4: Train-test split ──────────────────────────────────────────────────

# sets using stratified sampling"

X_train, X_test, y_train, y_test = train_test_split(
 X_encoded, y,
 test_size=TEST_SIZE,
 random_state=RANDOM_STATE,
 stratify=y # Preserves 84.5% / 15.5% ratio in both sets
)

print('=== TRAIN / TEST SPLIT ===')
print(f'Training set : {len(X_train):,} samples')
print(f'Test set : {len(X_test):,} samples')
print(f'\nTraining class distribution:')
print(f' Class 0 (no purchase) : {sum(y_train==0):,} ({sum(y_train==0)/len(y_train)*100:.1f}%)')
print(f' Class 1 (purchase) : {sum(y_train==1):,} ({sum(y_train==1)/len(y_train)*100:.1f}%)')

In [ ]:
# ── Step 5: Feature scaling ───────────────────────────────────────────────────

# (zero mean, unit variance) — prerequisite for Logistic Regression"
# IMPORTANT: Fit scaler ONLY on training data, then transform both sets
# (Fitting on test data = data leakage)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # Fit + transform on train
X_test_scaled = scaler.transform(X_test) # Transform only on test

print('StandardScaler applied.')
print(f'Training mean (first 3 features) : {X_train_scaled[:, :3].mean(axis=0).round(4)}')
print(f'Training std (first 3 features) : {X_train_scaled[:, :3].std(axis=0).round(4)}')

In [ ]:
# ── Step 6: SMOTE oversampling ────────────────────────────────────────────────

# minority-class samples... produces a balanced training set of
# approximately 16,600 samples. Critically, SMOTE is applied AFTER
# the train-test split to prevent data leakage."

smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print('=== AFTER SMOTE ===')
print(f'Training samples before SMOTE : {len(X_train_scaled):,}')
print(f'Training samples after SMOTE : {len(X_train_smote):,}')
print(f'\nClass distribution after SMOTE:')
print(f' Class 0 : {sum(y_train_smote==0):,} ({sum(y_train_smote==0)/len(y_train_smote)*100:.1f}%)')
print(f' Class 1 : {sum(y_train_smote==1):,} ({sum(y_train_smote==1)/len(y_train_smote)*100:.1f}%)')
print(f'\nTest set is UNCHANGED (no SMOTE applied to test) — {len(X_test_scaled):,} samples')

---
## Section 5 — Model Training
> **> *"L2 (Ridge) regularisation is applied with regularisation strength C=1.0... model parameters are estimated using the L-BFGS optimisation algorithm"*

In [ ]:
# ── Train Logistic Regression ────────────────────────────────────────────────
# Hyperparameters:
# - penalty = 'l2' (Ridge regularisation)
# - C = 1.0 (regularisation strength)
# - solver = 'lbfgs' (L-BFGS optimisation algorithm)
# - max_iter = 1000 (sufficient iterations to converge)

model = LogisticRegression(
 penalty='l2',
 C=1.0,
 solver='lbfgs',
 max_iter=1000,
 random_state=RANDOM_STATE
)

model.fit(X_train_smote, y_train_smote)

print('Logistic Regression model trained successfully.')
print(f'Number of features used : {model.n_features_in_}')
print(f'Number of iterations taken: {model.n_iter_[0]}')
print(f'Classes : {model.classes_}')

In [ ]:
# ── 5-Fold Stratified Cross Validation ───────────────────────────────────────
# Extra validation step to confirm model stability
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(model, X_train_smote, y_train_smote,
 cv=cv, scoring='roc_auc')

print('=== 5-FOLD CROSS VALIDATION (ROC-AUC) ===')
for i, score in enumerate(cv_scores, 1):
 print(f' Fold {i}: {score:.4f}')
print(f'\nMean AUC : {cv_scores.mean():.4f}')
print(f'Std AUC : {cv_scores.std():.4f}')
print('\n(Low std = model is stable, not overfitting to one split)')

---
## Section 6 — Evaluation & Results
> **> 

In [ ]:
# ── Predictions on test set ───────────────────────────────────────────────────
y_pred = model.predict(X_test_scaled)
y_pred_prob = model.predict_proba(X_test_scaled)[:, 1] # Probability of purchase

# ── Metrics ───────────────────────────────────────────────────────────────────
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)

print('=' * 55)
print(' MODEL PERFORMANCE ON TEST SET ')
print('=' * 55)
print(f' Accuracy : {accuracy:.4f} ')
print(f' ROC-AUC : {roc_auc:.4f} ')
print('=' * 55)
print()
print('=== CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred,
 target_names=['No Purchase (0)', 'Purchase (1)']))

In [ ]:
# ── Plot 6: Confusion Matrix ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
 display_labels=['No Purchase', 'Purchase'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Fig 6: Confusion Matrix — Logistic Regression\n',
 fontsize=12, fontweight='bold')

# Annotate with TP / FP / FN / TN labels
tn, fp, fn, tp = cm.ravel()
print(f'True Negatives (TN): {tn:,} — correctly predicted no purchase')
print(f'False Positives (FP): {fp:,} — predicted purchase, actually no purchase')
print(f'False Negatives (FN): {fn:,} — predicted no purchase, actually purchased')
print(f'True Positives (TP): {tp:,} — correctly predicted purchase')

plt.tight_layout()
plt.savefig('fig6_confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig6_confusion_matrix.png')

In [ ]:
# ── Plot 7: ROC Curve ─────────────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='#E07B54', lw=2.5,
 label=f'Logistic Regression (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5,
 label='Random Classifier (AUC = 0.500)')
ax.fill_between(fpr, tpr, alpha=0.1, color='#E07B54')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Fig 7: ROC Curve — Logistic Regression\n( ROC-AUC = 0.91)',
 fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig('fig7_roc_curve.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig7_roc_curve.png')

---
## Section 7 — Feature Interpretation
> **> *"PageValues = +2.84 (strongest positive), ExitRates = -1.92, Month_Nov = +1.47, BounceRates = -1.31"*

In [ ]:
# ── Extract and display model coefficients ───────────────────────────────────
# Top 10 features by absolute coefficient magnitude
feature_names = X_encoded.columns.tolist()
coefficients = model.coef_[0]

coef_df = pd.DataFrame({
 'Feature' : feature_names,
 'Coefficient': coefficients
}).sort_values('Coefficient', key=abs, ascending=False).reset_index(drop=True)

coef_df['Rank'] = coef_df.index + 1
coef_df['Direction'] = coef_df['Coefficient'].apply(
 lambda x: 'Increases purchase probability' if x > 0 else 'Decreases purchase probability'
)

print('=== TOP 15 FEATURES BY COEFFICIENT MAGNITUDE ===')
print(coef_df[['Rank', 'Feature', 'Coefficient', 'Direction']].head(15).to_string(index=False))

In [ ]:
# ── Plot 8: Top 20 feature coefficients ──────────────────────────────────────
top_n = 20
top_df = coef_df.head(top_n).sort_values('Coefficient')
colors = ['#E07B54' if c > 0 else '#5B8DB8' for c in top_df['Coefficient']]

fig, ax = plt.subplots(figsize=(9, 8))
bars = ax.barh(top_df['Feature'], top_df['Coefficient'],
 color=colors, edgecolor='white', linewidth=0.8)
ax.axvline(0, color='black', linewidth=0.8, linestyle='-')
ax.set_xlabel('Logistic Regression Coefficient (standardised)', fontsize=11)
ax.set_title(f'Fig 8: Top {top_n} Feature Coefficients\n'
 '(Orange = increases purchase probability | Blue = decreases)\n'
 'Top Features',
 fontsize=11, fontweight='bold')

# Legend
pos_patch = mpatches.Patch(color='#E07B54', label='Positive — predicts purchase')
neg_patch = mpatches.Patch(color='#5B8DB8', label='Negative — predicts no purchase')
ax.legend(handles=[pos_patch, neg_patch], loc='lower right')

plt.tight_layout()
plt.savefig('fig8_feature_coefficients.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig8_feature_coefficients.png')

In [ ]:
# ── Business Insights from Coefficients ──────────────────────────────────────
# (This section matches discussion)
print('=' * 60)
print(' BUSINESS INSIGHTS FROM MODEL COEFFICIENTS')
print(' ')
print('=' * 60)

insights = [
 ('PageValues',
 'STRONGEST positive predictor. Users visiting high-value product pages are '
 'much more likely to purchase. Prioritise showing these pages to users.'),
 ('ExitRates',
 'STRONGEST negative predictor. High exit rates signal disengagement. '
 'Reduce exit rates with better CTAs and page design.'),
 ('BounceRates',
 'Negative predictor. Users who bounce frequently are unlikely to purchase. '
 'Improve landing page relevance to reduce bounce.'),
 ('Month_Nov',
 'Positive predictor. November (Black Friday / Diwali season) drives '
 'significantly higher purchase intent.'),
 ('VisitorType_Returning_Visitor',
 'Positive predictor. Returning visitors convert at much higher rates. '
 'Invest in loyalty programmes and personalisation.'),
]

for feature, insight in insights:
 coef_val = coef_df[coef_df['Feature'] == feature]['Coefficient'].values
 coef_str = f'{coef_val[0]:+.3f}' if len(coef_val) > 0 else 'N/A'
 print(f'\n[{feature} | coef={coef_str}]')
 print(f' {insight}')

---
## Section 8 — Save Model & Summary

In [ ]:
# ── Save trained model and scaler ────────────────────────────────────────────
# ( modular pipeline, deployment ready)
joblib.dump(model, 'ubps_logistic_regression_model.pkl')
joblib.dump(scaler, 'ubps_scaler.pkl')

print('Model saved : ubps_logistic_regression_model.pkl')
print('Scaler saved : ubps_scaler.pkl')
print()

# ── List all output files ─────────────────────────────────────────────────────
output_files = [f for f in os.listdir('.') if f.endswith(('.png', '.pkl'))]
print('=== ALL OUTPUT FILES GENERATED ===')
for f in sorted(output_files):
 print(f' {f}')

In [ ]:
# ── Final Results Summary — matches Results exactly ───────────────────
from sklearn.metrics import precision_score, recall_score, f1_score

prec_macro = precision_score(y_test, y_pred, average='macro')
rec_macro = recall_score(y_test, y_pred, average='macro')
f1_macro = f1_score(y_test, y_pred, average='macro')
prec_weighted= precision_score(y_test, y_pred, average='weighted')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print()
print('╔══════════════════════════════════════════════════════╗')
print('║ FINAL RESULTS — USER BEHAVIOUR PREDICTION ║')
print('║ Logistic Regression on UCI Shoppers Dataset ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║ Accuracy : {accuracy:.4f} ║')
print(f'║ ROC-AUC : {roc_auc:.4f} ║')
print(f'║ Macro F1-Score : {f1_macro:.4f} ║')
print(f'║ Macro Precision : {prec_macro:.4f} ║')
print(f'║ Macro Recall : {rec_macro:.4f} ║')
print(f'║ Weighted F1-Score : {f1_weighted:.4f} ║')
print('╠══════════════════════════════════════════════════════╣')
print('║ Dataset : UCI Online Shoppers (12,330 records) ║')
print('║ Train/Test : 80% / 20% stratified ║')
print('║ Imbalance : SMOTE on training set only ║')
print('║ Model : LogisticRegression(C=1.0, l2, lbfgs) ║')
print('╚══════════════════════════════════════════════════════╝')

In [ ]:
# ── Optional: Predict on a new single session ────────────────────────────────
# This shows how the trained model is used in deployment
# (matches Prediction and Interpretation component)

def predict_user_session(session_dict, model, scaler, feature_columns):
 """
 Predict purchase probability for a single user session.

 Parameters:
 session_dict : dict of raw feature values (same format as dataset)
 model : trained LogisticRegression model
 scaler : fitted StandardScaler
 feature_columns: list of column names after one-hot encoding

 Returns:
 prediction : 0 (no purchase) or 1 (purchase)
 probability : probability of purchase
 """
 session_df = pd.DataFrame([session_dict])

 # Apply same one-hot encoding
 cat_cols = ['Month', 'VisitorType', 'Browser', 'Region', 'TrafficType', 'OperatingSystems']
 session_enc = pd.get_dummies(session_df, columns=[c for c in cat_cols if c in session_df.columns])
 session_enc['Weekend'] = session_enc['Weekend'].astype(int)

 # Align with training columns (add missing cols as 0)
 session_aligned = session_enc.reindex(columns=feature_columns, fill_value=0)

 # Scale and predict
 session_scaled = scaler.transform(session_aligned)
 prediction = model.predict(session_scaled)[0]
 probability = model.predict_proba(session_scaled)[0][1]

 return prediction, probability


# Example: High-intent user (high PageValues, returning visitor, November)
example_session = {
 'Administrative': 2, 'Administrative_Duration': 45.0,
 'Informational': 0, 'Informational_Duration': 0.0,
 'ProductRelated': 15, 'ProductRelated_Duration': 480.0,
 'BounceRates': 0.01, 'ExitRates': 0.02,
 'PageValues': 35.0, 'SpecialDay': 0.0,
 'Month': 'Nov', 'OperatingSystems': 2,
 'Browser': 2, 'Region': 1, 'TrafficType': 2,
 'VisitorType': 'Returning_Visitor', 'Weekend': False
}

pred, prob = predict_user_session(example_session, model, scaler, X_encoded.columns.tolist())
print('=== PREDICTION FOR EXAMPLE SESSION ===')
print(f'Prediction : {"PURCHASE" if pred == 1 else "NO PURCHASE"}')
print(f'Probability : {prob:.4f} ({prob*100:.1f}% chance of purchasing)')
print()
print('This user has high PageValues, is a Returning Visitor in November')
print('— model correctly identifies them as high purchase intent.')